## Collecting Data

In [1]:
import os
import pickle
from typing import Literal
import cv2
import time
import json
from landmarkers.inferences import Inference, InferenceSequence
from landmarkers.mp.hands import MPVideoLandmarker, MediapipeHandsMetadata
import numpy as np

# -------------------------
# Cargar configuración desde config.json
# -------------------------
with open('../config.json', 'r') as f:
    config = json.load(f)

collect_config = config['collect_data']
ACTIONS = collect_config['actions']
SEQUENCE_LENGTH = collect_config['sequence_length']
N_SEQUENCES = collect_config['n_sequences']
DATA_PATH = collect_config['data_path']
NUM_LANDMARKS = collect_config['num_landmarks']
COUNTDOWN = collect_config['countdown']
MODEL_PATH = collect_config['model_path']
NUM_HANDS = collect_config['num_hands']
WINDOW_WIDTH = collect_config['window_width']
WINDOW_HEIGHT = collect_config['window_height']

os.makedirs(DATA_PATH, exist_ok=True)
for action in ACTIONS:
    os.makedirs(os.path.join(DATA_PATH, action), exist_ok=True)

# -------------------------
# Funciones auxiliares
# -------------------------
def create_empty_hand() -> Inference:
    zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
    return Inference(
        landmarks=zeros,
        world_landmarks=zeros,
        metadata=MediapipeHandsMetadata(category_name="Right", index=0, score=0.0)
    )

def get_hand(inferences, category_name: Literal['Left', 'Right']) -> Inference:
    if not inferences:
        return create_empty_hand()
    hands = [inf for inf in inferences if inf.metadata.category_name == category_name]
    if hands:
        return hands[0]
    else:
        zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
        return Inference(
            landmarks=zeros,
            world_landmarks=zeros,
            metadata=MediapipeHandsMetadata(category_name=category_name, index=0, score=0.0)
        )

# -------------------------
# Grabación de secuencia
# -------------------------
def record_sequence(landmarker: MPVideoLandmarker, cap, action: str, seq_idx: int, sequence_length: int):
    window_name = f"{action}{seq_idx}"

    # Leer un frame para conocer tamaño
    ret, frame = cap.read()
    if not ret:
        return False

    h, w = frame.shape[:2]

    # Crear ventana escalada x3
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, WINDOW_WIDTH, WINDOW_HEIGHT)

    print(f"\nPreparado para grabar: {action} secuencia {seq_idx}. Presiona 's' para iniciar.")


    # -------------------------
    # Esperar a presionar 's' mostrando landmarks continuamente
    # -------------------------
    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        inferences = landmarker.infer(frame, int(time.time() * 1000))
        hand_right = get_hand(inferences, "Right")
        hand_left = get_hand(inferences, "Left")

        # Dibujar landmarks
        for lm in hand_right.landmarks.array:
            x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
            cv2.circle(frame, (x, y), 3, (0,255,0), -1)
        for lm in hand_left.landmarks.array:
            x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
            cv2.circle(frame, (x, y), 3, (0,0,255), -1)

        cv2.putText(frame, f"Presiona 's' para iniciar {action}{seq_idx}", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
        cv2.imshow(window_name, frame)

        key = cv2.waitKey(10) & 0xFF
        if key == ord('s'):
            break
        elif key == ord('q'):
            cv2.destroyWindow(window_name)
            return False

    # -------------------------
    # Cuenta atrás mostrando feed en tiempo real
    # -------------------------
    start_time = time.time()
    while True:
        elapsed = time.time() - start_time
        remaining = COUNTDOWN - int(elapsed)
        if remaining <= 0:
            break

        ret, frame = cap.read()
        if not ret:
            continue

        ts = int(time.time() * 1000)
        inferences = landmarker.infer(frame, ts)
        hand_right = get_hand(inferences, "Right")
        hand_left = get_hand(inferences, "Left")

        # Dibujar landmarks
        for lm in hand_right.landmarks.array:
            x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
            cv2.circle(frame, (x, y), 3, (0,255,0), -1)
        for lm in hand_left.landmarks.array:
            x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
            cv2.circle(frame, (x, y), 3, (0,0,255), -1)

        cv2.putText(frame, f"Comenzando en {remaining}...", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        cv2.imshow(window_name, frame)
        cv2.waitKey(1)

        # Imprimir solo una vez por segundo
        if int(elapsed) != int(elapsed - 0.05):
            print(f"{remaining}...")

    print("¡Grabando ahora!")

    # -------------------------
    # Inicializar secuencias
    # -------------------------
    seq_right = InferenceSequence(fixed_buffer_length=sequence_length)
    seq_left = InferenceSequence(fixed_buffer_length=sequence_length)
    frames_captured = 0

    while frames_captured < sequence_length:
        ret, frame = cap.read()
        if not ret:
            continue
        ts = int(time.time() * 1000)
        inferences = landmarker.infer(frame, ts)

        hand_right = get_hand(inferences, "Right")
        hand_left = get_hand(inferences, "Left")
        seq_right.append(hand_right, ts)
        seq_left.append(hand_left, ts)

        # Dibujar landmarks
        for lm in hand_right.landmarks.array:
            x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
            cv2.circle(frame, (x, y), 3, (0,255,0), -1)
        for lm in hand_left.landmarks.array:
            x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
            cv2.circle(frame, (x, y), 3, (0,0,255), -1)

        cv2.putText(frame, f"{action}{seq_idx} frame {frames_captured+1}/{sequence_length}",
                    (10,80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)
        cv2.imshow(window_name, frame)

        frames_captured += 1
        if cv2.waitKey(1) & 0xFF == ord('q'):
            cv2.destroyWindow(window_name)
            return False

    # Guardar secuencia completa en carpeta propia
    seq_folder = os.path.join(DATA_PATH, action, f"seq_{seq_idx}")
    os.makedirs(seq_folder, exist_ok=True)

    right_path = os.path.join(seq_folder, "right.pkl")
    left_path = os.path.join(seq_folder, "left.pkl")

    with open(right_path, "wb") as f:
        pickle.dump(seq_right, f)
    with open(left_path, "wb") as f:
        pickle.dump(seq_left, f)

    cv2.destroyWindow(window_name)
    print(f"Secuencia guardada en {seq_folder}")
    return True

# -------------------------
# Grabación de todas las acciones
# -------------------------
def record_actions(landmarker, cap):
    for action in ACTIONS:
        for seq_idx in range(N_SEQUENCES):
            success = record_sequence(landmarker, cap, action, seq_idx, SEQUENCE_LENGTH)
            if not success:
                print("Grabación interrumpida por el usuario")
                return

# -------------------------
# Main
# -------------------------
def main():
    cap = cv2.VideoCapture(0)
    try:
        with MPVideoLandmarker(model_path=MODEL_PATH, num_hands=NUM_HANDS) as landmarker:
            print("Iniciando grabación de secuencias...")
            record_actions(landmarker, cap)
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Grabación terminada")

if __name__ == "__main__":
    main()


2026-04-01 10:31:39.175933: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-01 10:31:39.231456: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-01 10:31:40.551647: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775032302.111141   65345 inference_feedback_manager.cc:114] Feedback

Iniciando grabación de secuencias...


W0000 00:00:1775032302.554682   65347 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.



Preparado para grabar: grab secuencia 0. Presiona 's' para iniciar.
2...
1...
¡Grabando ahora!
Secuencia guardada en ./dataset/grab/seq_0

Preparado para grabar: grab secuencia 1. Presiona 's' para iniciar.
2...
1...
¡Grabando ahora!
Secuencia guardada en ./dataset/grab/seq_1

Preparado para grabar: grab secuencia 2. Presiona 's' para iniciar.
2...
¡Grabando ahora!
Secuencia guardada en ./dataset/grab/seq_2

Preparado para grabar: grab secuencia 3. Presiona 's' para iniciar.
Grabación interrumpida por el usuario
Grabación terminada
